# 教師あり学習 — 回帰タスク

## データセットの読み込み
- カリフォルニア州の各地域ごとの住宅価格のデータセット  
  [https://inria.github.io/scikit-learn-mooc/python_scripts/datasets_california_housing.html](https://inria.github.io/scikit-learn-mooc/python_scripts/datasets_california_housing.html)

In [ ]:
# カリフォルニア州の各地域ごとの住宅価格のデータセット
# https://inria.github.io/scikit-learn-mooc/python_scripts/datasets_california_housing.html

from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()    # housing : Bunch ≒ dict
# housing.data          = housing['data']           : ndarray   : 説明変数（独立変数）
# housing.feature_names = housing['feature_names']  : list      : 変数名のリスト
# housing.target        = housing['target']         : ndarray   : 目的変数（従属変数）
# housing.DESCR         = housing['DESCR']          : string    : データセットの概要

# print(housing.DESCR)
print("読み込み終了")

説明変数：

    -  MedInc     世帯所得の中央値
    -  HouseAge   築年数の中央値
    -  AveRooms   部屋数の平均値
    -  AveBedrms  寝室数の平均値
    -  Population ブロックの人口
    -  AveOccup   世帯人数の平均値
    -  Latitude   ブロックの緯度
    -  Longitude  ブロックの経度

目的変数は住宅価格の中央値 (`MedHouseVal`，単位は十万ドル ($100,000)) である．  
この目的変数を，説明変数から推定するように学習させる．

In [ ]:
import pandas as pd

#　説明変数を DataFrame にして x に代入
x = pd.DataFrame(housing.data, columns=housing.feature_names)

#　目的変数を Series にして y に代入
y = pd.Series(housing.target, name='MedHouseVal')

# x,y を結合
df = pd.concat([x, y], axis=1)
display(df)

## データセットの概要

In [ ]:
# 各変数の基本統計量の確認
df.describe()

In [ ]:
# 各変数のヒストグラム
import matplotlib.pyplot as plt

# binsで棒の数を多くして，figsize でグラフの大きさを大きくしている
df.hist(bins=50, figsize=(10, 10))
plt.show()

In [ ]:
# 相関係数と散布図行列

# 相関係数表の表示 — 相関係数が1に近いほど強い正の相関，-1に近いほど強い負の相関，0に近いほど無相関
display(df.corr())

# 散布図行列の表示
from pandas.plotting import scatter_matrix
# figsize でグラフを大きくし，alpha でドットの透明度を上げている
sm = scatter_matrix(df, figsize=(18, 18), alpha=0.2)

## 準備 — 訓練データとテストデータの分割

<div style="margin-left:4em;">

| 変数      | 内容 |
|:--------- |:---- |
| `x_train` | 訓練データの説明変数 |
| `x_test`  | テストデータの説明変数 |
| `y_train` | 訓練データの目的変数 |
| `y_test`  | テストデータの目的変数 |
</div>

- 訓練データのサイズは全体の70%(0.7)，テストデータのサイズは全体の30%(0.3)


In [ ]:
# 学習データとテストデータの分割
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.7, test_size=0.3, random_state=0)

print("分割終了")

## 単回帰分析による学習
- `MedInc` だけを説明変数として単回帰分析をしてみる
  $$
  \mathrm{MedHouseVal} = A\cdot\mathrm{MedInc} + B\quad(A: 係数，B: 切片)
  $$
  というモデル

In [ ]:
# 単回帰分析とその結果

# 説明変数を MedInc のみに限定
x_train1 = x_train[["MedInc"]]
x_test1 = x_test[["MedInc"]]

# LinearRegression (回帰分析) モデルを生成して訓練データを適用
from sklearn.linear_model import LinearRegression
sr_model = LinearRegression()
sr_model.fit(x_train1, y_train)

# 結果の表示
print("切片 = ", sr_model.intercept_)
display(pd.DataFrame({"変数名": x_train1.columns, "係数": sr_model.coef_}))

In [ ]:
# グラフによる可視化
import matplotlib.pyplot as plt

plt.title("Regression Line")    # タイトル
plt.xlabel("MedInc")            # 横軸ラベル
plt.ylabel("MedHouseVal")       # 縦軸ラベル

plt.scatter(x_train1, y_train, s=1, alpha=0.2)    # 散布図(青)
plt.plot(x_train1['MedInc'], sr_model.predict(x_train1), color="red")  # 回帰直線(赤)

plt.show()  # 表示

In [ ]:
# 決定係数の表示
# 0 ～ 1 で，1に近いほど良い成績
print("r^2 (訓練データ):   ", sr_model.score(x_train1, y_train))
print("r^2 (テストデータ): ", sr_model.score(x_test1, y_test))

In [ ]:
# 訓練データとテストデータにおける，
# 観測値と予測値の二乗平均平方根誤差(RMSE; 二乗誤差の平均値の平方根)の表示

# x_train1 と x_test1 に対する予測値
sr_pred_train = sr_model.predict(x_train1)
sr_pred_test = sr_model.predict(x_test1)

# RMSEの表示
from sklearn.metrics import mean_squared_error
from math import sqrt
print("RMSE (訓練データ):   ", sqrt(mean_squared_error(y_train, sr_pred_train)))
print("RMSE (テストデータ): ", sqrt(mean_squared_error(y_test, sr_pred_test)))

## 重回帰分析による学習
- 説明変数すべてを使って重回帰分析をしてみる
  $$
  \mathrm{MedHouseVal}=A_0\cdot\mathrm{MedInc}+\cdots+A_7\cdot\mathrm{Longitude}+B
  \quad(A_0～A_7: 係数，B: 切片)
  $$
  というモデル

In [ ]:
# 重回帰分析とその結果

# 説明変数 (x_train) はすべて使用

# LinearRegression (回帰分析) モデルを生成して訓練データを適用
from sklearn.linear_model import LinearRegression
mr_model = LinearRegression()
mr_model.fit(x_train, y_train)

# 結果の表示
print("切片 = ", mr_model.intercept_)
display(pd.DataFrame({"変数名": x_train.columns, "係数": mr_model.coef_}))

In [ ]:
# 決定係数の表示
print("r^2 (訓練データ):   ", mr_model.score(x_train, y_train))
print("r^2 (テストデータ): ", mr_model.score(x_test, y_test))

In [ ]:
# 訓練データとテストデータにおける，
# 観測値と予測値の二乗平均平方根誤差(RMSE; 二乗誤差の平均値の平方根)の表示

# x_train1 と x_test1 に対する予測値
mr_pred_train = mr_model.predict(x_train)
mr_pred_test = mr_model.predict(x_test)

# RMSEの表示
from sklearn.metrics import mean_squared_error
from math import sqrt
print("RMSE (訓練データ):   ", sqrt(mean_squared_error(y_train, mr_pred_train)))
print("RMSE (テストデータ): ", sqrt(mean_squared_error(y_test, mr_pred_test)))

## FNNを用いた学習

In [ ]:
# FNNの学習を行う
from sklearn.neural_network import MLPRegressor
# hidden_layer_sizes で隠れ層のユニット数を250に増加
mlp_model = MLPRegressor(random_state=0, hidden_layer_sizes=(250,))
mlp_model.fit(x_train, y_train)

# 決定係数の表示
print("r^2 (訓練データ):   ", mlp_model.score(x_train, y_train))
print("r^2 (テストデータ): ", mlp_model.score(x_test, y_test))

In [ ]:
# 訓練データとテストデータにおける，
# 観測値と予測値の二乗平均平方根誤差(RMSE; 二乗誤差の平均値の平方根)の表示

# x_train1 と x_test1 に対する予測値
mlp_pred_train = mlp_model.predict(x_train)
mlp_pred_test = mlp_model.predict(x_test)

# RMSEの表示
from sklearn.metrics import mean_squared_error
from math import sqrt
print("RMSE (訓練データ):   ", sqrt(mean_squared_error(y_train, mlp_pred_train)))
print("RMSE (テストデータ): ", sqrt(mean_squared_error(y_test, mlp_pred_test)))

## ランダムフォレストを用いた学習

In [ ]:
# ランダムフォレストの学習を行う
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(random_state=0)
rf_model.fit(x_train, y_train)

# 決定係数の表示
print("r^2 (訓練データ):   ", rf_model.score(x_train, y_train))
print("r^2 (テストデータ): ", rf_model.score(x_test, y_test))

In [ ]:
# 訓練データとテストデータにおける，
# 観測値と予測値の二乗平均平方根誤差(RMSE; 二乗誤差の平均値の平方根)の表示

# x_train1 と x_test1 に対する予測値
rf_pred_train = rf_model.predict(x_train)
rf_pred_test = rf_model.predict(x_test)

# 二乗平均平方根誤差(RMSE) — 二乗誤差の平均値の平方根
from sklearn.metrics import mean_squared_error
from math import sqrt
print("RMSE (訓練データ):   ", sqrt(mean_squared_error(y_train, rf_pred_train)))
print("RMSE (テストデータ): ", sqrt(mean_squared_error(y_test, rf_pred_test)))

## 結果のまとめ

In [ ]:
print("単回帰分析:")
print("    訓練データ:")
print("        r^2:  ", sr_model.score(x_train1, y_train))
print("        RMSE: ", sqrt(mean_squared_error(y_train, sr_pred_train)))
print("    テストデータ:")
print("        r^2:  ", sr_model.score(x_test1, y_test))
print("        RMSE: ", sqrt(mean_squared_error(y_test, sr_pred_test)))

print("重回帰分析:")
print("    訓練データ:")
print("        r^2:  ", mr_model.score(x_train, y_train))
print("        RMSE: ", sqrt(mean_squared_error(y_train, mr_pred_train)))
print("    テストデータ:")
print("        r^2:  ", mr_model.score(x_test, y_test))
print("        RMSE: ", sqrt(mean_squared_error(y_test, mr_pred_test)))

print("FNN:")
print("    訓練データ:")
print("        r^2:  ", mlp_model.score(x_train, y_train))
print("        RMSE: ", sqrt(mean_squared_error(y_train, mlp_pred_train)))
print("    テストデータ:")
print("        r^2:  ", mlp_model.score(x_test, y_test))
print("        RMSE: ", sqrt(mean_squared_error(y_test, mlp_pred_test)))

print("ランダムフォレスト:")
print("    訓練データ:")
print("        r^2:  ", rf_model.score(x_train, y_train))
print("        RMSE: ", sqrt(mean_squared_error(y_train, rf_pred_train)))
print("    テストデータ:")
print("        r^2:  ", rf_model.score(x_test, y_test))
print("        RMSE: ", sqrt(mean_squared_error(y_test, rf_pred_test)))
